In [70]:
import pandas as pd
total_deltas = pd.read_csv('total_deltas.csv')
shap_matrix = pd.read_csv('shap_matrix.csv')
prediction_results = pd.read_csv('prediction_results_by_hadm.csv')

total_data_full = pd.read_parquet('ind_hosp.parquet')
correct_predictions = prediction_results[prediction_results['correct_prediction'] == True]
test_pairs = correct_predictions[['subject_id', 'hadm_id']].drop_duplicates()
test_pairs_tuple = list(zip(test_pairs['subject_id'], test_pairs['hadm_id']))

total_data = total_data_full[
    total_data_full[['subject_id', 'hadm_id']].apply(tuple, axis=1).isin(test_pairs_tuple)
].copy()

print(f"Data filtered to test hospitalizations only:")
print(f"  Total rows: {len(total_data)}")
print(f"  Unique patients: {total_data['subject_id'].nunique()}")
print(f"  Unique hospitalizations: {total_data['hadm_id'].nunique()}")

Data filtered to test hospitalizations only:
  Total rows: 237684
  Unique patients: 33311
  Unique hospitalizations: 58385


In [71]:
agg_dict = {}

for col in total_data.columns:
    if col in ['subject_id', 'hadm_id']:
        continue
    
    if col.startswith('icd_'):
        agg_dict[col] = 'max'
    
    elif col.startswith('ccsr_'):
        agg_dict[col] = 'max'
    
    elif col.startswith('lab_') and col.endswith('_daily'):
        agg_dict[col] = 'mean'
    
    elif col in ['cumulative_procedures', 'cumulative_medications']:
        agg_dict[col] = 'mean'
    
    elif col in ['num_medications_daily', 'num_procedures_daily']:
        agg_dict[col] = 'mean'
    
    elif col in ['gender_male', 'age', 'los', 'num_diagnoses', 'num_chronic', 
                 'comorbidity_score', 'prior_admissions_12mo', 
                 'admission_type', 'discharge_location', 'insurance', 'race',
                 'admission_location']:
        agg_dict[col] = 'first'
    
    elif col == 'target_readmission_30d':
        agg_dict[col] = 'max'
    
    else:
        agg_dict[col] = 'first'

print(f"Aggregating {len(agg_dict)} columns...")
total_data = total_data.groupby(['subject_id', 'hadm_id']).agg(agg_dict).reset_index()
total_data.head(5)

Aggregating 136 columns...


,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,admission_location_EMERGENCY ROOM,admission_location_INFORMATION NOT AVAILABLE,admission_location_INTERNAL TRANSFER TO OR FROM PSYCH,admission_location_PACU,admission_location_PHYSICIAN REFERRAL,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,gender_male
0,10000764,27897940,2132-10-19,4,86,1,2132-10-14,76,32,1,...,False,False,False,False,False,False,True,False,False,1
1,10001492,27463908,2136-09-25,1,71,1,2136-09-23,4,4,0,...,True,False,False,False,False,False,False,False,False,0
2,10001667,22672901,2173-08-24,1,86,1,2173-08-22,13,6,1,...,False,False,False,False,False,False,True,False,False,0
3,10001725,25563031,2110-04-14,2,46,1,2110-04-11,36,18,1,...,False,False,False,True,False,False,False,False,False,0
4,10001860,21441082,2188-03-30,3,84,1,2188-03-27,36,9,1,...,True,False,False,False,False,False,False,False,False,1


In [72]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [73]:
def get_patient_analysis(subject_id, hadm_id):
    patient_delta = total_deltas[
        (total_deltas['subject_id'] == subject_id) & 
        (total_deltas['hadm_id'] == hadm_id)
    ].copy()
    print(f"Actual risk: {patient_delta['risk_actual_mean'].iloc[0]:.4f}")
    patient_shap = shap_matrix[shap_matrix['hadm_id'] == hadm_id].copy()
    shap_features = patient_shap.drop('hadm_id', axis=1).T
    shap_features.columns = ['shap_value']
    shap_features['feature'] = shap_features.index
    
    combined = patient_delta.merge(shap_features, on='feature', how='inner')
    combined['delta_abs'] = combined['delta_mean'].abs()
    combined['shap_abs'] = combined['shap_value'].abs()
    
    return {
        'patient_id': subject_id,
        'hadm_id': hadm_id,
        'n_days': patient_delta['los'].iloc[0],
        'risk_actual': patient_delta['risk_actual_mean'].iloc[0],
        'combined': combined,
        'by_delta': combined.sort_values('delta_abs', ascending=False),
        'by_shap': combined.sort_values('shap_abs', ascending=False)
    }

def interpret_delta(delta, mode_direction):
    if mode_direction == 'add':
        if delta > 0:
            return "without → ↑ risk"
        elif delta < 0:
            return "without → ↓ risk"

    elif mode_direction == 'remove':
        if delta < 0:
            return "with → ↓ risk"
        elif delta > 0:
            return "with → ↑ risk"

    elif mode_direction == 'increase_to_mean':
        if delta < 0:
            return "lower than mean → ↓ risk"
        elif delta > 0:
            return "lower than mean → ↑ risk"
    
    elif mode_direction == 'reduce_to_mean':
        if delta < 0:
            return "higher than mean → ↓ risk"
        elif delta > 0:
            return "higher than mean → ↑ risk"
    
    elif mode_direction == 'low':
        if delta > 0:
            return "low → ↑ risk"
        elif delta < 0:
            return "low → ↓ risk"

    elif mode_direction == 'high':
        if delta > 0:
            return "high → ↑ risk"
        elif delta < 0:
            return "high → ↓ risk"
    
    elif mode_direction == 'normal':
        if delta > 0:
            return "normal range → ↑ risk"
        elif delta < 0:
            return "normal range → ↓ risk"

    elif mode_direction == 'older':
        if delta < 0:
            return "older → ↓ risk"
        elif delta > 0:
            return "older → ↑ risk"
    
    elif mode_direction == 'younger':
        if delta > 0:
            return "younger → ↑ risk"
        elif delta < 0:
            return "younger → ↓ risk"

    elif mode_direction == 'opposite':
        if delta > 0:
            return "↑ risk"
        elif delta < 0:
            return "↓ risk"

In [74]:
patient_info = total_data.copy()
icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

patient_info['icd_count'] = patient_info[icd_cols].sum(axis=1)
patient_info['ccsr_count'] = patient_info[ccsr_cols].sum(axis=1)
patient_info['lab_count'] = patient_info[lab_cols].notna().sum(axis=1)

patient_info = patient_info[
    (patient_info['icd_count'] >= 1) & (patient_info['ccsr_count'] >= 1) &
    (patient_info['lab_count'] >= 1)
].copy()

age_groups = {
    'young': (18, 40),
    'middle': (40, 65),
    'elderly': (65, 80),
    'very_elderly': (80, 120)
}

def get_age_group(age):
    for group, (low, high) in age_groups.items():
        if low <= age < high:
            return group
    return 'unknown'

patient_info = patient_info[['subject_id', 'hadm_id', 'gender_male', 'age', 'los', 'icd_count', 'ccsr_count', 'lab_count', 'target_readmission_30d']]
patient_info['age_group'] = patient_info['age'].apply(get_age_group)

print(f"Patient info created:")
print(f"  Total hospitalizations: {len(patient_info)}")
print(f"  With readmission: {patient_info['target_readmission_30d'].sum():.0f}")
print(f"  Without readmission: {len(patient_info) - patient_info['target_readmission_30d'].sum():.0f}")

def get_stratified_sample(patient_info, sample_size=20, random_state=42):
    readmission_yes = patient_info[patient_info['target_readmission_30d'] == 1]
    readmission_no = patient_info[patient_info['target_readmission_30d'] == 0]
    
    n_total = sample_size
    n_yes = n_total // 2
    n_no = n_total - n_yes
    samples = []

    for group, label, n in [(readmission_yes, 'with_readmission', n_yes), 
                            (readmission_no, 'without_readmission', n_no)]:
       
        age_groups_list = group['age_group'].unique()
        n_per_age = n // len(age_groups_list) if len(age_groups_list) > 0 else 0
        remainder = n - n_per_age * len(age_groups_list)
        
        for i, age_group in enumerate(age_groups_list):
            age_group_patients = group[group['age_group'] == age_group]

            n_age = n_per_age + (1 if i < remainder else 0)
            if n_age == 0:
                continue

            male = age_group_patients[age_group_patients['gender_male'] == 1]
            female = age_group_patients[age_group_patients['gender_male'] == 0]
            
            n_male_age = min(n_age // 2, len(male))
            n_female_age = n_age - n_male_age
            
            if len(male) < n_male_age:
                n_male_age = len(male)
                n_female_age = n_age - n_male_age
            
            if len(female) < n_female_age:
                n_female_age = len(female)
                n_male_age = n_age - n_female_age
            
            if len(male) > 0 and n_male_age > 0:
                sample_male = male.sample(n=min(n_male_age, len(male)), random_state=random_state)
                samples.append(sample_male)
            
            if len(female) > 0 and n_female_age > 0:
                sample_female = female.sample(n=min(n_female_age, len(female)), random_state=random_state)
                samples.append(sample_female)

    if len(samples) > 0:
        sample = pd.concat(samples).sample(frac=1, random_state=random_state)
    else:
        print("No samples selected")
        return pd.DataFrame()
    
    print(f"\nSelected {len(sample)} hospitalizations")
    print(f"   With readmission: {sample['target_readmission_30d'].sum():.0f}")
    print(f"   Without readmission: {len(sample) - sample['target_readmission_30d'].sum():.0f}")
    print(f"   Male: {sample['gender_male'].sum():.0f}")
    print(f"   Female: {len(sample) - sample['gender_male'].sum():.0f}")
    
    print(f"\nAge groups in sample:")
    print(sample['age_group'].value_counts().sort_index())
    
    return sample

SAMPLE_SIZE = 16
sample_patients_df = get_stratified_sample(patient_info, sample_size=SAMPLE_SIZE, random_state=42)
sample_patients_df

Patient info created:
  Total hospitalizations: 34892
  With readmission: 6498
  Without readmission: 28394

Selected 16 hospitalizations
   With readmission: 8
   Without readmission: 8
   Male: 8
   Female: 8

Age groups in sample:
age_group
elderly         4
middle          4
very_elderly    4
young           4
Name: count, dtype: int64


,subject_id,hadm_id,gender_male,age,los,icd_count,ccsr_count,lab_count,target_readmission_30d,age_group
9380,11618742,28081676,1,76,9,3.0,9,20,1,elderly
56263,19622090,25925869,0,75,4,3.0,3,20,1,elderly
924,10164170,24771340,0,87,5,3.0,4,20,1,very_elderly
15166,12592821,29232336,1,30,6,1.0,2,20,0,young
45388,17780345,26440784,0,71,3,1.0,5,20,0,elderly
15279,12610089,21423469,0,60,7,3.0,3,20,0,middle
9846,11698781,20902547,1,88,3,1.0,2,20,0,very_elderly
38937,16668434,23499682,0,80,6,2.0,3,20,0,very_elderly
1738,10301630,26672817,1,57,23,6.0,10,20,1,middle
47097,18056725,25708121,0,32,4,2.0,3,20,0,young


In [75]:
def get_patient_full_data_with_analysis(subject_id, hadm_id, total_data):

    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) & 
        (total_data['hadm_id'] == hadm_id)
    ]

    patient = patient_raw.iloc[0].to_dict()
    clean_data = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'age': patient.get('age'),
        'gender': 'Male' if patient.get('gender_male') == 1 else 'Female',
        'los': patient.get('los'),
        'readmission': 'Yes' if patient.get('target_readmission_30d') == 1 else 'No',
    }
    
    icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
    icd_present = [col for col in icd_cols if col in patient and patient[col] == 1]
    
    ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
    ccsr_present = [col for col in ccsr_cols if col in patient and patient[col] == 1]
    
    lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]
    lab_present = {col: patient[col] for col in lab_cols if col in patient and pd.notna(patient[col])}
    
    clean_data['icd_diagnoses'] = [map_feature_name(col) for col in icd_present]
    clean_data['icd_count'] = len(icd_present)
    
    clean_data['ccsr_diagnoses'] = [map_feature_name(col) for col in ccsr_present]
    clean_data['ccsr_count'] = len(ccsr_present)
    
    clean_data['lab_values'] = {map_feature_name(col): val for col, val in lab_present.items()}
    clean_data['lab_count'] = len(lab_present)
    
    other_features = ['num_diagnoses', 'num_chronic', 'comorbidity_score',
                      'num_medications_daily', 'prior_admissions_12mo',
                      'cumulative_procedures', 'cumulative_medications',
                      'num_procedures_daily']
    
    for feat in other_features:
        if feat in patient and pd.notna(patient[feat]):
            clean_data[feat] = patient[feat]
    
    race_cols = [col for col in total_data.columns if col.startswith('race_')]
    for col in race_cols:
        if col in patient and patient[col] == 1:
            clean_data['race'] = col.replace('race_', '').replace('_', ' ').title()
            break
    if 'race' not in clean_data:
        clean_data['race'] = 'Unknown'
    
    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            clean_data['insurance'] = col.replace('insurance_', '')
            break
    if 'insurance' not in clean_data:
        clean_data['insurance'] = 'Unknown'
    
    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            adm_type = col.replace('admission_type_', '').replace('_', ' ').title()
            clean_data['admission_type'] = adm_type
            break
    if 'admission_type' not in clean_data:
        clean_data['admission_type'] = 'Unknown'
    
    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            discharge = col.replace('discharge_location_', '').replace('_', ' ').title()
            clean_data['discharge_location'] = discharge
            break
    if 'discharge_location' not in clean_data:
        clean_data['discharge_location'] = 'Unknown'
    
    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            adm_loc = col.replace('admission_location_', '').replace('_', ' ').title()
            clean_data['admission_location'] = adm_loc
            break
    if 'admission_location' not in clean_data:
        clean_data['admission_location'] = 'Unknown'

    result = get_patient_analysis(subject_id, hadm_id)
    if result:
        existing_features_original = icd_present + ccsr_present + list(lab_present.keys())
        result['combined'] = result['combined'][
            result['combined']['feature'].isin(existing_features_original)
        ]
        
        result['by_delta'] = result['combined'].sort_values('delta_abs', ascending=False)
        result['by_shap'] = result['combined'].sort_values('shap_abs', ascending=False)
        
        result['combined']['feature_display'] = result['combined']['feature'].apply(map_feature_name)
        result['by_delta']['feature_display'] = result['by_delta']['feature'].apply(map_feature_name)
        result['by_shap']['feature_display'] = result['by_shap']['feature'].apply(map_feature_name)
        
        if len(result['by_delta']) > 0:
            result['by_delta']['interpretation'] = result['by_delta'].apply(
                lambda row: interpret_delta(row['delta_mean'], row['direction_mode']), axis=1
            )
        
        if len(result['by_shap']) > 0:
            result['by_shap']['interpretation'] = result['by_shap'].apply(
                lambda row: interpret_delta(row['delta_mean'], row['direction_mode']), axis=1
            )
        clean_data['analysis'] = result
    
    return clean_data


def print_patient_full_with_analysis(clean_data):
    if clean_data is None:
        return
        
    print(f"\nDEMOGRAPHICS:")
    print(f"  Age: {clean_data['age']}")
    print(f"  Gender: {clean_data['gender']}")
    print(f"  Race: {clean_data.get('race', 'Unknown')}")
    print(f"  Insurance: {clean_data.get('insurance', 'Unknown')}")
    print(f"  LOS: {clean_data['los']} days")
    print(f"  Readmission: {clean_data['readmission']}")

    print(f"\nADMISSION DETAILS:")
    print(f"  Admission Type: {clean_data.get('admission_type', 'Unknown')}")
    print(f"  Admission Location: {clean_data.get('admission_location', 'Unknown')}")
    print(f"  Discharge Location: {clean_data.get('discharge_location', 'Unknown')}")
    
    if clean_data['icd_count'] > 0:
        print(f"\nICD DIAGNOSES ({clean_data['icd_count']}):")
        for dx in clean_data['icd_diagnoses']:
            print(f"  - {dx}")
    
    if clean_data['ccsr_count'] > 0:
        print(f"\nCCSR CATEGORIES ({clean_data['ccsr_count']}):")
        for ccsr in clean_data['ccsr_diagnoses']:
            print(f"  - {ccsr}")
    
    if clean_data['lab_count'] > 0:
        print(f"\nLAB VALUES ({clean_data['lab_count']}):")
        for lab, value in clean_data['lab_values'].items():
            print(f"  - {lab}: {value:.2f}")
    
    other_feats = ['num_diagnoses', 'num_chronic', 'comorbidity_score',
                   'num_medications_daily', 'prior_admissions_12mo',
                   'cumulative_procedures', 'cumulative_medications',
                   'num_procedures_daily']
    
    print(f"\nOTHER FEATURES:")
    for feat in other_feats:
        if feat in clean_data:
            print(f"  {feat}: {clean_data[feat]:.2f}")
    
    if 'analysis' in clean_data and clean_data['analysis']:
        result = clean_data['analysis']
        
        print(f"\nDELTA/SHAP:")
        if len(result['by_delta']) > 0:
            print(result['by_delta'][['feature_display', 'delta_mean', 'shap_value', 'interpretation']].to_string(index=False))

### Middle 

In [76]:
SUBJECT_ID = 10301630
HADM_ID = 26672817
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.3520

DEMOGRAPHICS:
  Age: 57
  Gender: Male
  Race: White
  Insurance: Private
  LOS: 23 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Clinic Referral
  Discharge Location: Home Health Care

ICD DIAGNOSES (6):
  - Essential (primary) hypertension (I10)
  - Gastroesophageal reflux disease without esophagitis (K219)
  - Personal history of nicotine dependence (Z87891)
  - Atherosclerotic heart disease of native coronary artery without angina pectoris (I2510)
  - Acute kidney failure, unspecified (N179)
  - Anemia, unspecified (D649)

CCSR CATEGORIES (10):
  - Personal/family history of disease (FAC021)
  - Other specified status (FAC025)
  - Fluid and electrolyte disorders (END011)
  - Coronary atherosclerosis and other heart disease (CIR011)
  - Disorders of lipid metabolism (END010)
  - Essential hypertension (CIR007)
  - Esophageal disorders (DIG004)
  - Aplastic anemia (BLD003)
  - External cause codes: place of 

In [77]:
SUBJECT_ID = 10692997
HADM_ID = 24362796
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.1249

DEMOGRAPHICS:
  Age: 61
  Gender: Male
  Race: White
  Insurance: Medicare
  LOS: 2 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Surgical Same Day Admission
  Admission Location: Physician Referral
  Discharge Location: Home

ICD DIAGNOSES (4):
  - Essential (primary) hypertension (I10)
  - Gastroesophageal reflux disease without esophagitis (K219)
  - Personal history of nicotine dependence (Z87891)
  - Type 2 diabetes mellitus without complications (E119)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Essential hypertension (CIR007)
  - Esophageal disorders (DIG004)

LAB VALUES (20):
  - Anion Gap (50868): 13.00
  - Bicarbonate (50882): 27.00
  - Calcium (50893): 9.10
  - Chloride (50902): 100.00
  - Creatinine (50912): 1.00
  - Glucose (50931): 188.00
  - Magnesium (50960): 2.20
  - Phosphate (50970): 2.60
  - Potassium (50971): 4.10
  - Sodium (50983): 136.00
  - BUN (51006): 11.00
  - Hematocrit (51221): 36.50
  - Hem

In [78]:
SUBJECT_ID = 12610089
HADM_ID = 21423469
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.1124

DEMOGRAPHICS:
  Age: 60
  Gender: Female
  Race: Other
  Insurance: Private
  LOS: 7 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Transfer From Hospital
  Discharge Location: Home

ICD DIAGNOSES (3):
  - Essential (primary) hypertension (I10)
  - Personal history of nicotine dependence (Z87891)
  - Type 2 diabetes mellitus without complications (E119)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Essential hypertension (CIR007)
  - Aplastic anemia (BLD003)

LAB VALUES (20):
  - Anion Gap (50868): 12.71
  - Bicarbonate (50882): 22.57
  - Calcium (50893): 8.73
  - Chloride (50902): 102.29
  - Creatinine (50912): 0.66
  - Glucose (50931): 135.14
  - Magnesium (50960): 2.16
  - Phosphate (50970): 2.71
  - Potassium (50971): 3.91
  - Sodium (50983): 137.57
  - BUN (51006): 9.57
  - Hematocrit (51221): 31.40
  - Hemoglobin (51222): 10.30
  - MCH (51248): 28.56
  - MCHC (51249): 32.81
  - 

In [79]:
SUBJECT_ID = 19424524
HADM_ID = 26254319
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.4641

DEMOGRAPHICS:
  Age: 58
  Gender: Female
  Race: White
  Insurance: Private
  LOS: 4 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Emergency Room
  Discharge Location: Home Health Care

ICD DIAGNOSES (3):
  - Personal history of nicotine dependence (Z87891)
  - Acute kidney failure, unspecified (N179)
  - Anxiety disorder, unspecified (F419)

CCSR CATEGORIES (4):
  - Personal/family history of disease (FAC021)
  - Fluid and electrolyte disorders (END011)
  - Aplastic anemia (BLD003)
  - Acute and unspecified renal failure (GEN002)

LAB VALUES (20):
  - Anion Gap (50868): 15.00
  - Bicarbonate (50882): 18.75
  - Calcium (50893): 8.57
  - Chloride (50902): 106.25
  - Creatinine (50912): 1.40
  - Glucose (50931): 93.00
  - Magnesium (50960): 1.82
  - Phosphate (50970): 2.97
  - Potassium (50971): 4.08
  - Sodium (50983): 136.00
  - BUN (51006): 18.00
  - Hematocrit (51221): 23.15
  - Hemoglobin (51222): 7.72
  - MCH (5124

### Young

In [80]:
SUBJECT_ID = 15794497
HADM_ID = 25969239	
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.5623

DEMOGRAPHICS:
  Age: 37
  Gender: Female
  Race: Black/African American
  Insurance: Medicaid
  LOS: 6 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Walk-In/Self Referral
  Discharge Location: Home Health Care

ICD DIAGNOSES (1):
  - Long term (current) use of anticoagulants (Z7901)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Other specified status (FAC025)
  - Aplastic anemia (BLD003)

LAB VALUES (20):
  - Anion Gap (50868): 10.50
  - Bicarbonate (50882): 26.00
  - Calcium (50893): 8.72
  - Chloride (50902): 99.50
  - Creatinine (50912): 0.50
  - Glucose (50931): 98.83
  - Magnesium (50960): 1.70
  - Phosphate (50970): 3.77
  - Potassium (50971): 3.57
  - Sodium (50983): 136.00
  - BUN (51006): 8.00
  - Hematocrit (51221): 24.27
  - Hemoglobin (51222): 7.88
  - MCH (51248): 31.58
  - MCHC (51249): 32.48
  - MCV (51250): 97.50
  - Platelet Count (51265): 137.67
  - RDW (51277): 16.18
  - R

In [81]:
SUBJECT_ID = 12592821
HADM_ID = 29232336
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.0364

DEMOGRAPHICS:
  Age: 30
  Gender: Male
  Race: Unknown
  Insurance: UNKNOWN
  LOS: 6 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Emergency Room
  Discharge Location: Home

ICD DIAGNOSES (1):
  - Essential (primary) hypertension (I10)

CCSR CATEGORIES (2):
  - Personal/family history of disease (FAC021)
  - Essential hypertension (CIR007)

LAB VALUES (20):
  - Anion Gap (50868): 17.80
  - Bicarbonate (50882): 22.60
  - Calcium (50893): 9.22
  - Chloride (50902): 102.40
  - Creatinine (50912): 0.72
  - Glucose (50931): 81.20
  - Magnesium (50960): 2.14
  - Phosphate (50970): 2.46
  - Potassium (50971): 3.80
  - Sodium (50983): 138.80
  - BUN (51006): 9.00
  - Hematocrit (51221): 40.54
  - Hemoglobin (51222): 13.68
  - MCH (51248): 29.90
  - MCHC (51249): 33.82
  - MCV (51250): 88.60
  - Platelet Count (51265): 236.60
  - RDW (51277): 13.88
  - RBC (51279): 4.58
  - WBC (51301): 11.00

OTHER FEATURES:
  num_diagnoses: 48

In [82]:
SUBJECT_ID = 11011770
HADM_ID = 28546869
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.4253

DEMOGRAPHICS:
  Age: 30
  Gender: Male
  Race: White
  Insurance: Private
  LOS: 2 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Emergency Room
  Discharge Location: Home

ICD DIAGNOSES (4):
  - Gastroesophageal reflux disease without esophagitis (K219)
  - Major depressive disorder, single episode, unspecified (F329)
  - Anxiety disorder, unspecified (F419)
  - Urinary tract infection, site not specified (N390)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Esophageal disorders (DIG004)
  - External cause codes: place of occurrence of the external cause (EXT027)

LAB VALUES (17):
  - Anion Gap (50868): 14.00
  - Bicarbonate (50882): 29.00
  - Chloride (50902): 103.00
  - Creatinine (50912): 0.70
  - Glucose (50931): 75.00
  - Potassium (50971): 3.60
  - Sodium (50983): 142.00
  - BUN (51006): 4.00
  - Hematocrit (51221): 37.20
  - Hemoglobin (51222): 12.30
  - MCH (51248): 28.10
  -

In [83]:
SUBJECT_ID = 18056725
HADM_ID = 25708121
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.1193

DEMOGRAPHICS:
  Age: 32
  Gender: Female
  Race: White
  Insurance: Private
  LOS: 4 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Emergency Room
  Discharge Location: Home

ICD DIAGNOSES (2):
  - Personal history of nicotine dependence (Z87891)
  - Anemia, unspecified (D649)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Fluid and electrolyte disorders (END011)
  - Aplastic anemia (BLD003)

LAB VALUES (20):
  - Anion Gap (50868): 10.67
  - Bicarbonate (50882): 22.67
  - Calcium (50893): 7.53
  - Chloride (50902): 105.00
  - Creatinine (50912): 0.63
  - Glucose (50931): 128.00
  - Magnesium (50960): 2.17
  - Phosphate (50970): 2.60
  - Potassium (50971): 3.60
  - Sodium (50983): 134.67
  - BUN (51006): 4.00
  - Hematocrit (51221): 26.93
  - Hemoglobin (51222): 9.33
  - MCH (51248): 29.27
  - MCHC (51249): 34.80
  - MCV (51250): 84.00
  - Platelet Count (51265): 252.00
  - RDW (51277): 12.90
  

### Very elderly

In [84]:
SUBJECT_ID = 10164170
HADM_ID = 24771340
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.2068

DEMOGRAPHICS:
  Age: 87
  Gender: Female
  Race: White - Other European
  Insurance: Medicare
  LOS: 5 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Urgent
  Admission Location: Transfer From Hospital
  Discharge Location: Home Health Care

ICD DIAGNOSES (3):
  - Atherosclerotic heart disease of native coronary artery without angina pectoris (I2510)
  - Unspecified atrial fibrillation (I4891)
  - Urinary tract infection, site not specified (N390)

CCSR CATEGORIES (4):
  - Personal/family history of disease (FAC021)
  - Coronary atherosclerosis and other heart disease (CIR011)
  - Heart failure (CIR019)
  - Cardiac dysrhythmias (CIR017)

LAB VALUES (20):
  - Anion Gap (50868): 11.00
  - Bicarbonate (50882): 29.40
  - Calcium (50893): 8.56
  - Chloride (50902): 100.80
  - Creatinine (50912): 0.72
  - Glucose (50931): 81.60
  - Magnesium (50960): 1.82
  - Phosphate (50970): 3.12
  - Potassium (50971): 3.82
  - Sodium (50983): 137.40
  - BUN (51006): 15

In [85]:
SUBJECT_ID = 11698781
HADM_ID = 20902547
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.0712

DEMOGRAPHICS:
  Age: 88
  Gender: Male
  Race: White
  Insurance: Medicare
  LOS: 3 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Transfer From Hospital
  Discharge Location: Skilled Nursing Facility

ICD DIAGNOSES (1):
  - Personal history of nicotine dependence (Z87891)

CCSR CATEGORIES (2):
  - Personal/family history of disease (FAC021)
  - External cause codes: place of occurrence of the external cause (EXT027)

LAB VALUES (20):
  - Anion Gap (50868): 13.50
  - Bicarbonate (50882): 21.50
  - Calcium (50893): 9.00
  - Chloride (50902): 102.00
  - Creatinine (50912): 0.95
  - Glucose (50931): 115.50
  - Magnesium (50960): 1.90
  - Phosphate (50970): 2.20
  - Potassium (50971): 3.80
  - Sodium (50983): 137.00
  - BUN (51006): 20.50
  - Hematocrit (51221): 41.25
  - Hemoglobin (51222): 13.05
  - MCH (51248): 31.35
  - MCHC (51249): 31.65
  - MCV (51250): 99.00
  - Platelet Count (51265): 115.00
  - RDW (51277)

In [86]:
SUBJECT_ID = 16668434
HADM_ID = 23499682
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.1015

DEMOGRAPHICS:
  Age: 80
  Gender: Female
  Race: White
  Insurance: Medicare
  LOS: 6 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Transfer From Hospital
  Discharge Location: Hospice

ICD DIAGNOSES (2):
  - Essential (primary) hypertension (I10)
  - Urinary tract infection, site not specified (N390)

CCSR CATEGORIES (3):
  - Other specified status (FAC025)
  - Essential hypertension (CIR007)
  - External cause codes: place of occurrence of the external cause (EXT027)

LAB VALUES (20):
  - Anion Gap (50868): 10.80
  - Bicarbonate (50882): 20.40
  - Calcium (50893): 9.14
  - Chloride (50902): 108.40
  - Creatinine (50912): 1.12
  - Glucose (50931): 135.00
  - Magnesium (50960): 2.24
  - Phosphate (50970): 2.88
  - Potassium (50971): 4.00
  - Sodium (50983): 139.60
  - BUN (51006): 17.80
  - Hematocrit (51221): 32.38
  - Hemoglobin (51222): 10.02
  - MCH (51248): 29.28
  - MCHC (51249): 30.98
  - MCV (51250): 94.60
  - P

In [87]:
SUBJECT_ID = 19972786
HADM_ID = 29611193
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.2264

DEMOGRAPHICS:
  Age: 81
  Gender: Male
  Race: Black/African American
  Insurance: Medicare
  LOS: 4 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Ew Emer.
  Admission Location: Emergency Room
  Discharge Location: Home Health Care

ICD DIAGNOSES (1):
  - Hyperlipidemia, unspecified (E785)

CCSR CATEGORIES (7):
  - Personal/family history of disease (FAC021)
  - Other specified status (FAC025)
  - Fluid and electrolyte disorders (END011)
  - Disorders of lipid metabolism (END010)
  - Diabetes mellitus with complication (END003)
  - Heart failure (CIR019)
  - Hypertension with complications and secondary hypertension (CIR008)

LAB VALUES (20):
  - Anion Gap (50868): 13.33
  - Bicarbonate (50882): 27.67
  - Calcium (50893): 9.73
  - Chloride (50902): 93.67
  - Creatinine (50912): 1.13
  - Glucose (50931): 118.33
  - Magnesium (50960): 1.97
  - Phosphate (50970): 3.33
  - Potassium (50971): 4.57
  - Sodium (50983): 130.33
  - BUN (51006): 15.67
  - Hem

### Elderly

In [88]:
SUBJECT_ID = 19622090
HADM_ID = 25925869
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.3821

DEMOGRAPHICS:
  Age: 75
  Gender: Female
  Race: White
  Insurance: Medicare
  LOS: 4 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Emergency Room
  Discharge Location: Home Health Care

ICD DIAGNOSES (3):
  - Gastroesophageal reflux disease without esophagitis (K219)
  - Type 2 diabetes mellitus without complications (E119)
  - Urinary tract infection, site not specified (N390)

CCSR CATEGORIES (3):
  - Personal/family history of disease (FAC021)
  - Heart failure (CIR019)
  - Esophageal disorders (DIG004)

LAB VALUES (20):
  - Anion Gap (50868): 12.67
  - Bicarbonate (50882): 29.33
  - Calcium (50893): 8.70
  - Chloride (50902): 102.67
  - Creatinine (50912): 1.30
  - Glucose (50931): 150.00
  - Magnesium (50960): 2.17
  - Phosphate (50970): 4.13
  - Potassium (50971): 4.13
  - Sodium (50983): 140.67
  - BUN (51006): 36.00
  - Hematocrit (51221): 24.57
  - Hemoglobin (51222): 8.00
  - MCH (51248): 31.20
  - 

In [89]:
SUBJECT_ID = 11618742
HADM_ID = 28081676
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.4399

DEMOGRAPHICS:
  Age: 76
  Gender: Male
  Race: White
  Insurance: Medicare
  LOS: 9 days
  Readmission: Yes

ADMISSION DETAILS:
  Admission Type: Direct Emer.
  Admission Location: Physician Referral
  Discharge Location: Home

ICD DIAGNOSES (3):
  - Hyperlipidemia, unspecified (E785)
  - Acute kidney failure, unspecified (N179)
  - Long term (current) use of insulin (Z794)

CCSR CATEGORIES (9):
  - Other specified status (FAC025)
  - Coronary atherosclerosis and other heart disease (CIR011)
  - Disorders of lipid metabolism (END010)
  - Diabetes mellitus with complication (END003)
  - Heart failure (CIR019)
  - Cardiac dysrhythmias (CIR017)
  - Hypertension with complications and secondary hypertension (CIR008)
  - External cause codes: place of occurrence of the external cause (EXT027)
  - Acute and unspecified renal failure (GEN002)

LAB VALUES (20):
  - Anion Gap (50868): 14.22
  - Bicarbonate (50882): 27.00
  - Calcium (50893): 8.88
  - Chloride (50902): 100.1

In [90]:
SUBJECT_ID = 17780345
HADM_ID = 26440784
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.1174

DEMOGRAPHICS:
  Age: 71
  Gender: Female
  Race: White
  Insurance: Medicare
  LOS: 3 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Transfer From Hospital
  Discharge Location: Home

ICD DIAGNOSES (1):
  - Atherosclerotic heart disease of native coronary artery without angina pectoris (I2510)

CCSR CATEGORIES (5):
  - Other specified status (FAC025)
  - Fluid and electrolyte disorders (END011)
  - Coronary atherosclerosis and other heart disease (CIR011)
  - Heart failure (CIR019)
  - Hypertension with complications and secondary hypertension (CIR008)

LAB VALUES (20):
  - Anion Gap (50868): 12.00
  - Bicarbonate (50882): 28.00
  - Calcium (50893): 9.05
  - Chloride (50902): 102.50
  - Creatinine (50912): 0.50
  - Glucose (50931): 106.00
  - Magnesium (50960): 1.90
  - Phosphate (50970): 3.40
  - Potassium (50971): 4.50
  - Sodium (50983): 142.50
  - BUN (51006): 8.00
  - Hematocrit (51221): 36.95
  - Hemoglobi

In [91]:
SUBJECT_ID = 14507136
HADM_ID = 28573487
clean_data = get_patient_full_data_with_analysis(
    SUBJECT_ID, HADM_ID, total_data)
print_patient_full_with_analysis(clean_data)

Actual risk: 0.0498

DEMOGRAPHICS:
  Age: 66
  Gender: Male
  Race: Unknown
  Insurance: Medicare
  LOS: 4 days
  Readmission: No

ADMISSION DETAILS:
  Admission Type: Observation Admit
  Admission Location: Physician Referral
  Discharge Location: Rehab

ICD DIAGNOSES (4):
  - Essential (primary) hypertension (I10)
  - Hyperlipidemia, unspecified (E785)
  - Gastroesophageal reflux disease without esophagitis (K219)
  - Personal history of nicotine dependence (Z87891)

CCSR CATEGORIES (5):
  - Personal/family history of disease (FAC021)
  - Disorders of lipid metabolism (END010)
  - Essential hypertension (CIR007)
  - Esophageal disorders (DIG004)
  - External cause codes: place of occurrence of the external cause (EXT027)

LAB VALUES (20):
  - Anion Gap (50868): 13.00
  - Bicarbonate (50882): 28.00
  - Calcium (50893): 8.70
  - Chloride (50902): 98.67
  - Creatinine (50912): 0.80
  - Glucose (50931): 115.33
  - Magnesium (50960): 2.00
  - Phosphate (50970): 3.47
  - Potassium (50971):